In [8]:
import pandas as pd
import numpy as np
import os


In [11]:
DATA_DIR = "Data/"
OUTPUT_FILE = "consolidated_dataset.csv"

FILES = {
    "shock": "Shock_Data.csv",
    "cpi_values": "CPI_Values.csv",
    "cpi_weights": "CPI_Weights.csv",
    "exchange_rate": "Exchange_Rate.csv",
    "market_prices": "Market_Prices.csv",
    "population": "Population_Data.csv",
    "climate": "Climate_Data.csv",
    "socio_demographic": "Socio-Demographic Indicators.csv"
}


In [ ]:
def load_data(key, **kwargs):
    path = os.path.join(DATA_DIR, FILES[key])
    if path.endswith(".csv"):
        return pd.read_csv(path, **kwargs)
    elif path.endswith(".xlsx"):
        return pd.read_excel(path, **kwargs)
    else:
        raise ValueError("Unsupported file format")


def process_population(df):
    df = df.rename(columns={'Region': 'adm1_name', 'Population': 'Population'})

    # Map pop regions → market ADM1 regions
    mapping = {
        "Lower Shabelle": "Shabelle Hoose",
        "Lower Juba": "Juba Hoose",
        "Middle Shabelle": "Shabelle Dhexe",
        "Middle Juba": "Juba Dhexe",
        "North Mudug": "Mudug",
        "South Mudug": "Mudug",
        "Awdal": "Awdal",
        "Bakool": "Bakool",
        "Banadir": "Banadir",
        "Bari": "Bari",
        "Bay": "Bay",
        "Galgaduud": "Galgaduud",
        "Gedo": "Gedo",
        "Hiraan": "Hiraan",
        "Sanaag": "Sanaag",
        "Sool": "Sool",
        "Togdheer": "Togdheer",
        "Woqooyi Galbeed": "Woqooyi Galbeed"
    }

    df['adm1_name'] = df['adm1_name'].map(mapping)

    df['Date'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + df['Month'].astype(str),
        format='%Y-%B', errors='coerce'
    )

    df = df[df['Date'].dt.year.between(2015, 2025)].copy()

    return df[['Date', 'adm1_name', 'Population']]



def process_climate(df):
    df = df.rename(columns={
        'date': 'Date', 
        'region': 'mkt_name',
        'temperature': 'Climate_Temperature',
        'precipitation': 'Climate_Precipitation',
        'humidity': 'Climate_Humidity'
    })
    
    df['Date'] = pd.to_datetime(df['Date'])
    
    df = df[df['Date'].dt.year.between(2015, 2025)].copy()
    
    return df[['Date', 'mkt_name', 'Climate_Temperature', 'Climate_Precipitation', 'Climate_Humidity']]


def process_shock(df):
    df = df.rename(columns={
        'year': 'Year',
        'month': 'Month',
        'indicator': 'Indicator',
        'grouping': 'Grouping',
        'value': 'Value'
    })

    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

    df['Date'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + df['Month'].astype(str),
        format='%Y-%m'
    )

    df = df[df['Date'].dt.year.between(2015, 2025)].copy()

    df['Indicator_Group'] = (
        df['Indicator'].str.replace(" ", "_") + "_" +
        df['Grouping'].str.replace(" ", "_")
    )

    out = df.pivot_table(
        index="Date",
        columns="Indicator_Group",
        values="Value",
        aggfunc='first'
    ).reset_index()

    col_to_drop = ["GLM_Population"]
    out = out.drop(columns=col_to_drop, errors='ignore')

    out.columns.name = None
    return out



def process_cpi(values_df, weights_df):
    w = weights_df.copy()
    w["Indicator"] = (
        w["indicator"]
        .str.replace(", weights", "", regex=False)
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(",", "")
    )

    w_unique = (
        w.groupby("Indicator")["Value"]
         .last()
         .reset_index()
         .rename(columns={"Value": "Weight"})
    )

    df = values_df.copy()
    df["Indicator"] = (
        df["indicator"]
        .str.replace(" ", "_")
        .str.replace(",", "")
    )

    df["Date"] = (
        df["Date"]
        .str.replace("M", "-")
        .str.replace(r"-(\d)$", r"-0\1", regex=True)
    )
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")
    df = df[df['Date'].dt.year.between(2015, 2025)].copy()
    
    wide = df.pivot_table(
        index="Date",
        columns="Indicator",
        values="Value",
        aggfunc="mean"
    ).reset_index()

    weight_dict = dict(zip(w_unique["Indicator"], w_unique["Weight"]))

    for col in wide.columns:
        if col == "Date":
            continue

        if col in weight_dict:
            weight = weight_dict[col] / 100
            wide[col] = wide[col] * weight
        else:
            wide = wide.drop(columns=[col]) 

    return wide



def process_exchange(df):
    df["Date"] = pd.to_datetime(
        df["Date"].str.replace("M", "-"),
        format="%Y-%m"
    )
    df = df[df["Date"].dt.year.between(2015, 2025)].copy()
    return df[["Date", "Value"]].rename(columns={"Value": "Exchange_Rate"})



def process_market_prices(df):    
    id_cols = ["adm1_name", "adm2_name", "mkt_name", "price_date"]
    
    price_cols = {
        "c_maize": "c_maize_price",
        "c_rice": "c_rice_price",
        "c_sorghum": "c_sorghum_price",
        "c_oil": "c_oil_price",
        "c_food_price_index": "c_food_price_index"
    }
    
    # Select only the necessary columns
    all_cols = id_cols + list(price_cols.keys())
    existing = [c for c in df.columns if c in all_cols]
    df = df[existing].copy()

    df = df.rename(columns={"price_date": "Date", **price_cols})

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    df = df[df["Date"].dt.year.between(2015, 2025)].copy()

    price_vars = list(price_cols.values())
    for col in price_vars:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["Month"] = df["Date"].dt.to_period("M")
    
    group_cols = ["adm1_name", "adm2_name", "mkt_name", "Month"]
    out = df.groupby(group_cols)[price_vars].mean().reset_index()

    out["Date"] = out["Month"].dt.to_timestamp()

    return out.drop(columns=["Month"])



def process_socio_demo(df):
    df["Date"] = pd.to_datetime(df["Date"], format="%Y")
    df = df[df["Date"].dt.year.between(2015, 2025)].copy()

    out = df.pivot_table(
        index="Date",
        columns="indicator",
        values="Value"
    ).reset_index()

    out.columns = [
        c.replace(" ", "_") if c != "Date" else c
        for c in out.columns
    ]
    return out

In [16]:
def consolidate_data():        
    # Population
    print("Processing Population Data...")
    pop_df = load_data("population")
    pop_df_processed = process_population(pop_df)
    
    # Climate
    print("Processing Climate Data...")
    climate_df = load_data("climate") 
    climate_df_processed = process_climate(climate_df)
    
    pass
        
    # Market Prices
    print("Processing Market Prices Data...")
    market_df = load_data("market_prices")
    market_df_processed = process_market_prices(market_df)
    
    #  Master DataFrame
    
    master_df = market_df_processed.copy()
    
    master_df = pd.merge(
        master_df, 
        pop_df_processed, 
        on=['Date', 'adm1_name'], 
        how='left'
    )
    
    # Merge Climate onto the Master DataFrame (on Date and mkt_name)
    master_df = pd.merge(
        master_df, 
        climate_df_processed, 
        on=['Date', 'mkt_name'], 
        how='left'
    )
    
    # Load and process national, monthly data (Shock, CPI, Exchange Rate)
    
    # Shock
    print("Processing Shock Data...")
    shock_df = load_data("shock")
    shock_df_processed = process_shock(shock_df)
    master_df = pd.merge(master_df, shock_df_processed, on='Date', how='left')
    
    # CPI
    print("Processing CPI Data...")
    cpi_values_df = load_data("cpi_values")
    cpi_weights_df = load_data("cpi_weights")
    cpi_df_processed = process_cpi(cpi_values_df, cpi_weights_df)
    master_df = pd.merge(master_df, cpi_df_processed, on='Date', how='left')
    
    # Exchange Rate
    print("Processing Exchange Rate Data...")
    exchange_df = load_data("exchange_rate")
    exchange_df_processed = process_exchange(exchange_df)
    master_df = pd.merge(master_df, exchange_df_processed, on='Date', how='left')
    
    # Load and process national, annual data (Socio-Demographic)
    
    # Socio-Demographic
    print("Processing Socio-Demographic Data...")
    socio_df = load_data("socio_demographic")
    socio_df_processed = process_socio_demo(socio_df)
    
    # Merge annual data.
    master_df['Year_Date'] = master_df['Date'].dt.to_period('Y').dt.to_timestamp()
    
    master_df = pd.merge(
        master_df, 
        socio_df_processed, 
        left_on='Year_Date', 
        right_on='Date', 
        how='left', 
        suffixes=('', '_Socio')
    ).drop(columns=['Date_Socio', 'Year_Date'])
        
    master_df = master_df.sort_values(by=['adm1_name', 'adm2_name', 'mkt_name', 'Date']).reset_index(drop=True)
    
    output_path = os.path.join(DATA_DIR, OUTPUT_FILE)
    master_df.to_csv(output_path, index=False)
    
    print(f"Consolidation complete. Final dataset saved to: {output_path}")
    print(f"Final dataset shape: {master_df.shape}")
    
    return master_df



In [18]:
consolidate_data()


Processing Population Data...
Processing Climate Data...
Processing Market Prices Data...
Processing Shock Data...
Processing CPI Data...
Processing Exchange Rate Data...
Processing Socio-Demographic Data...
Consolidation complete. Final dataset saved to: Data/consolidated_dataset.csv
Final dataset shape: (6277, 55)


,adm1_name,adm2_name,mkt_name,c_maize_price,c_rice_price,c_sorghum_price,c_oil_price,c_food_price_index,Date,Population,...,MISCELLANEOUS_GOODS_AND_SERVICES,RECREATION_AND_CULTURE,RESTAURANTS_AND_HOTELS,TRANSPORT,Exchange_Rate,"GDP_per_capita,_US_Dollars",Government_final_consumption,Gross_fixed_capital_formation,Household_final_consumption,Minus:_Imports_of_goods_and_services
0,Awdal,Borama,Borama,4000.00,4649.66,2646.02,34010.71,0.56,2015-01-01,NaN,...,2.804937,0.997394,1.854169,3.988219,22285.0,414.263094,324289767.7,674627947.1,7.281761e+09,3.987255e+09
1,Awdal,Borama,Borama,4000.00,4616.42,2635.47,33206.32,0.55,2015-02-01,305455.0,...,2.804937,0.975330,1.854169,3.950615,22196.0,414.263094,324289767.7,674627947.1,7.281761e+09,3.987255e+09
2,Awdal,Borama,Borama,4000.00,4500.00,2573.14,33318.08,0.55,2015-03-01,305455.0,...,2.808122,1.001521,1.864399,3.950615,22211.0,414.263094,324289767.7,674627947.1,7.281761e+09,3.987255e+09
3,Awdal,Borama,Borama,4000.00,4500.00,2625.00,33135.40,0.55,2015-04-01,305455.0,...,2.810152,1.007315,1.864399,3.969913,22232.0,414.263094,324289767.7,674627947.1,7.281761e+09,3.987255e+09
4,Awdal,Borama,Borama,4000.00,4264.24,2094.63,32390.78,0.53,2015-05-01,305455.0,...,2.816379,1.006065,1.864399,3.862924,22265.0,414.263094,324289767.7,674627947.1,7.281761e+09,3.987255e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6272,Woqooyi Galbeed,Hargeysa,Hargeysa,4567.43,8138.72,3395.79,44072.87,0.75,2025-07-01,1492507.0,...,NaN,NaN,NaN,NaN,29949.0,NaN,NaN,NaN,NaN,NaN
6273,Woqooyi Galbeed,Hargeysa,Hargeysa,4478.42,8009.42,3354.89,44237.87,0.75,2025-08-01,1492507.0,...,NaN,NaN,NaN,NaN,29895.0,NaN,NaN,NaN,NaN,NaN
6274,Woqooyi Galbeed,Hargeysa,Hargeysa,4392.60,8021.06,3320.05,43928.03,0.74,2025-09-01,1492507.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6275,Woqooyi Galbeed,Hargeysa,Hargeysa,4413.05,8063.61,3324.17,43807.89,0.74,2025-10-01,1492507.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
